In [ ]:
%load_ext autoreload 
%autoreload 2
%matplotlib inline
import sys
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
from scipy.stats import gaussian_kde
import json
import os
import re
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import sim_loader as sim_load
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
import matplotlib.lines as mlines 
from matplotlib.animation import FuncAnimation
from matplotlib import pyplot as plt, ticker as mticker
import sim_loader as sim_load
import h5py
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

## Maximum mass of giant planets through core accretion as a function of stellar mass
The idea is to predict what is the scaling with stellar mass of the maximum size of giant planets, that should be observable by GAIA DR4

In [ ]:
# ============================================================
# FILE SORTING
# ============================================================

def extract_initial_time(filename):
    with h5py.File(filename, 'r') as f:
        time = f['simulation']['time'][()]
        if np.ndim(time) > 0:
            return float(time[0])
        return float(time)


# ============================================================
# LOAD SIMULATIONS 
# ============================================================
folder_path_multi = ["sims/opacities/1Msun/photoevap/Mdot_scatter/GAIA/02Msun/",
                    "sims/opacities/1Msun/photoevap/Mdot_scatter/GAIA/05Msun/",
                     "sims/opacities/1Msun/photoevap/Mdot_scatter/GAIA/08Msun/",
                     "sims/opacities/1Msun/photoevap/Mdot_scatter/GAIA/1Msun/",
                     "sims/opacities/1Msun/photoevap/Mdot_scatter/GAIA/1_3Msun/",
                     "sims/opacities/1Msun/photoevap/Mdot_scatter/GAIA/1_7Msun/"]

simulations_multi = []
sim_parameters_multi = []
parameters_multi = []

for folder in folder_path_multi:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_multi.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_multi.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_multi.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6,6))
color = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']  # Colors for each folder
# plotting the simulations for each folder
for i, folder in enumerate(folder_path_multi):
    simulations = simulations_multi[i]
    sim_parameters = sim_parameters_multi[i]
    parameters = parameters_multi[i]
    col = color[i]
    # position vs mass, color coded by sim
    for sim in simulations:
        axs.scatter(sim.position, sim.mass, s=10, alpha=0.5, color = col)
axs.set_xscale('log')
axs.set_yscale('log')
axs.set_xlabel('Position (AU)')
axs.set_ylabel('Mass (Earth Masses)')



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import astropy.units as u

# ============================================================
# STELLAR MASSES (from folder order)
# ============================================================
stellar_masses = np.array([0.2, 0.5, 0.8, 1.0, 1.3, 1.7])  # Msun

# ============================================================
# EXTRACT FINAL PLANET MASSES (in Jupiter masses) PER STELLAR MASS
# sim.mass is indexed by planet: sim.mass[i] = full time series of planet i
# so final mass of planet i = sim.mass[i][-1]
# ============================================================
giant_masses_by_mstar = []

for sims in simulations_multi:
    final_masses_mjup = []
    for sim in sims:
        for planet_track in sim.mass:              # loop over planets in this sim
            final_mass = planet_track[-1]            # last timestep of THIS planet
            final_mass_q = u.Quantity(final_mass)
            final_masses_mjup.append(final_mass_q.to(u.Mjup).value)

    final_masses_mjup = np.array(final_masses_mjup, dtype=float)
    giants = final_masses_mjup[final_masses_mjup > 1.0]   # M_p > M_jup
    giant_masses_by_mstar.append(giants)

for m_star, giants in zip(stellar_masses, giant_masses_by_mstar):
    print(f"M_star = {m_star} Msun: {len(giants)} giants out of up to 1000 sims")

# ============================================================
# VIOLIN PLOT (KDE done in log10-mass space, mean shown)
# ============================================================
fig, ax = plt.subplots(figsize=(6, 5))

positions, data_for_violin = [], []
for m_star, giants in zip(stellar_masses, giant_masses_by_mstar):
    if len(giants) >= 2:
        positions.append(m_star)
        data_for_violin.append(np.log10(giants))  # KDE in log-space

if data_for_violin:
    parts = ax.violinplot(data_for_violin, positions=positions,
                           widths=0.08, showmeans=True, showextrema=True)
    for body in parts['bodies']:
        body.set_facecolor("crimson")
        body.set_edgecolor("black")
        body.set_alpha(0.3)
    parts['cmeans'].set_color("black")
    parts['cmeans'].set_linewidth(2)
    # print the mean mass for each stellar mass 
    print("Mean giant planet masses (in Mjup) for each stellar mass:")
    for m_star, giants in zip(stellar_masses, giant_masses_by_mstar):
        if len(giants) > 0:
            mean_mass = np.mean(giants)
            print(f"M_star = {m_star} Msun: Mean M_p = {mean_mass:.2f} Mjup")
        else:
            print(f"M_star = {m_star} Msun: No giant planets")

# scatter overlay of individual planets (log10 space, same axis)
for m_star, giants in zip(stellar_masses, giant_masses_by_mstar):
    if len(giants) > 0:
        jitter = np.random.normal(0, 0.01, size=len(giants))
        ax.scatter(np.full_like(giants, m_star) + jitter, np.log10(giants),
                   color="crimson", alpha=0.08, s=6, zorder=3)

ax.axhline(0.0, color="gray", ls="--", lw=1, label=r"$M_p = M_{\rm Jup}$")  # log10(1)=0
ax.set_xlabel(r"Stellar mass $M_\star \, [M_\odot]$")
ax.set_ylabel(r"Giant planet mass $M_p \, [M_{\rm Jup}]$")

# relabel y-axis ticks from log10 back to linear Mjup values
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{10**y:.0f}"))

ax.legend()
ax.set_title(r"Distribution of giant planet masses ($M_p > M_{\rm Jup}$) vs. stellar mass")

plt.tight_layout()
plt.savefig("figures/giant_planet_mass_distribution_vs_mstar.png", dpi=200)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

# number of simulations per stellar mass (the denominator)
n_sims = np.array([sum(len(sim.mass) > 0 for sim in sims) for sims in simulations_multi])
# if every folder counts as one star regardless of planets formed, use instead:
# n_sims = np.array([len(sims) for sims in simulations_multi])

# common log-spaced bins for all stellar masses
all_giants = np.concatenate(giant_masses_by_mstar)
edges = np.logspace(0, np.log10(all_giants.max()), 12)   # 1 Mjup to max
centers = np.sqrt(edges[:-1] * edges[1:])                # geometric bin centers
dlog = np.diff(np.log10(edges))

norm = Normalize(vmin=stellar_masses.min(), vmax=stellar_masses.max())
cmap = cm.viridis

fig, ax = plt.subplots(figsize=(7, 5))
for m_star, giants, n in zip(stellar_masses, giant_masses_by_mstar, n_sims):
    counts, _ = np.histogram(giants, bins=edges)
    eta = counts / (n * dlog)
    err = np.sqrt(counts) / (n * dlog)
    ok = counts > 0                                      # log axis cannot show zeros
    ax.errorbar(centers[ok], eta[ok], yerr=err[ok], fmt="o", ms=6,
                color=cmap(norm(m_star)), capsize=2, alpha=0.9)

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=ax, label=r"$M_\star\ [M_\odot]$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$M_p\ [M_{\rm Jup}]$")
ax.set_ylabel(r"$\eta = \frac{1}{N_\star}\frac{dN}{d\log_{10} M_p}$")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from scipy.interpolate import PchipInterpolator

# number of simulations per stellar mass (the denominator)
n_sims = np.array([len(sims) for sims in simulations_multi])

# common log-spaced bins for all stellar masses
all_giants = np.concatenate(giant_masses_by_mstar)
N_BINS = 11                                              # increase for finer bins
edges = np.logspace(0, np.log10(all_giants.max()), N_BINS + 1)    # 1 Mjup to max
centers = np.sqrt(edges[:-1] * edges[1:])                # geometric bin centers
dlog = np.diff(np.log10(edges))

norm = Normalize(vmin=stellar_masses.min(), vmax=stellar_masses.max())
cmap = cm.viridis

fig, ax = plt.subplots(figsize=(7, 5))
for m_star, giants, n in zip(stellar_masses, giant_masses_by_mstar, n_sims):
    counts, _ = np.histogram(giants, bins=edges)
    eta = counts / (n * dlog)
    ok = counts > 0                                      # log axis cannot show zeros
    x, y = centers[ok], eta[ok]
    color = cmap(norm(m_star))

    ax.scatter(x, y, s=40, color=color, zorder=3)

    if len(x) >= 2:                                      # need 2+ points to interpolate
        f = PchipInterpolator(np.log10(x), np.log10(y))
        xx = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
        ax.plot(xx, 10 ** f(np.log10(xx)), color=color, lw=1.5, alpha=0.8, zorder=2)

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=ax, label=r"$M_\star\ [M_\odot]$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$M_p\ [M_{\rm Jup}]$")
ax.set_ylabel(r"$\eta = \frac{1}{N_\star}\frac{dN}{d\log_{10} M_p}$")
plt.savefig("figures/giant_planet_mass_distribution_vs_mstar_interpolated_12bins.png", dpi=200)